# Header census - every series, headers only

Reads **one DICOM header per series** for all training and test series, plus the
slice count, and writes `header_census.csv`. **CPU only, no GPU quota.** Headers are
read with `stop_before_pixels`, so this touches kilobytes per series, not the 570 GB.

Answers questions a one-study sample could only raise:

- Which tags survive anonymisation, and does it vary by vendor?
- Is the `Laterality` tag present everywhere? If so, laterality is solved.
- **How many series does the organisers' `Fluid_Sensitive` flag misfile?** A local
  sample found a T2 TSE (TE 91 ms, no fat-sat) labelled fluid=0.
- Does `PatientID` repeat across studies? If so, CV folds must group by patient.
- Vendor, field strength, transfer syntax and slices-per-series distributions.

**Settings:** accelerator None, internet off. Attach the competition data only.
**Output:** `header_census.csv` - contains UIDs, so keep it out of any public repo.


In [ ]:
import os, glob, time, json, collections
import numpy as np, pandas as pd, pydicom

T0 = time.time()
TIME_BUDGET_S = 8 * 3600
ROOT = None
for cand in ["/kaggle/input/rsna-knee-abnormality-detection",
             "/kaggle/input/competitions/rsna-knee-abnormality-detection"]:
    if os.path.exists(cand + "/train.csv"):
        ROOT = cand; break
assert ROOT, "attach the competition dataset"
print("root:", ROOT)

S = pd.concat([pd.read_csv(ROOT + "/train_series.csv").assign(split="train"),
               pd.read_csv(ROOT + "/test_series.csv").assign(split="test")], ignore_index=True)
print(len(S), "series")

FIELDS = ["Manufacturer", "ManufacturerModelName", "MagneticFieldStrength", "Laterality",
          "ImageLaterality", "PatientID", "PatientSex", "SeriesDescription", "ScanningSequence",
          "SequenceVariant", "ScanOptions", "MRAcquisitionType", "EchoTime", "RepetitionTime",
          "InversionTime", "FlipAngle", "EchoTrainLength", "SliceThickness",
          "SpacingBetweenSlices", "PixelSpacing", "Rows", "Columns", "BitsStored",
          "PhotometricInterpretation", "ContrastBolusAgent", "PixelBandwidth"]

def val(d, k):
    v = d.get(k, None)
    if v is None: return None
    if isinstance(v, (pydicom.multival.MultiValue, list, tuple)): return "|".join(str(x) for x in v)
    return str(v)


In [ ]:
rows, tagsets, failed = [], collections.Counter(), 0
for i, r in enumerate(S.itertuples(index=False)):
    if time.time() - T0 > TIME_BUDGET_S:
        print("time budget hit at", i); break
    sdir = f"{ROOT}/{r.split}_series/{r.StudyInstanceUID}/{r.SeriesInstanceUID}"
    files = sorted(glob.glob(sdir + "/*.dcm"))
    rec = {"split": r.split, "StudyInstanceUID": r.StudyInstanceUID,
           "SeriesInstanceUID": r.SeriesInstanceUID, "plane": r.Anatomical_Plane,
           "csv_fs": r.Fat_Suppression, "csv_fluid": r.Fluid_Sensitive, "n_slices": len(files),
           "bytes_per_slice": os.path.getsize(files[len(files)//2]) if files else None}
    if files:
        try:
            d = pydicom.dcmread(files[len(files)//2], stop_before_pixels=True)
            tagsets[frozenset(e.keyword for e in d if e.keyword)] += 1
            rec["TransferSyntaxUID"] = str(d.file_meta.get("TransferSyntaxUID", ""))
            for k in FIELDS: rec[k] = val(d, k)
        except Exception as e:
            failed += 1; rec["error"] = f"{type(e).__name__}: {e}"
    rows.append(rec)
    if (i + 1) % 2000 == 0:
        print(f"  {i+1}/{len(S)}  {time.time()-T0:.0f}s", flush=True)
H = pd.DataFrame(rows)
H.to_csv("/kaggle/working/header_census.csv", index=False)
print(f"wrote header_census.csv  {H.shape}  failed={failed}  {time.time()-T0:.0f}s")


In [ ]:
pct = lambda s: f"{100*s.mean():.1f}%"
print("=== tag survival across series ===")
all_tags = collections.Counter()
for ts, n in tagsets.items():
    for t in ts: all_tags[t] += n
N = sum(tagsets.values())
print(f"distinct tags {len(all_tags)} | present in every series {sum(v==N for v in all_tags.values())}")
print("present in <100% of series:", {k: f"{100*v/N:.0f}%" for k, v in sorted(all_tags.items(), key=lambda x: x[1]) if v < N})

print("\n=== laterality ===")
print("Laterality tag present:", pct(H.Laterality.notna()), "| values:", H.Laterality.value_counts(dropna=False).head(5).to_dict())
per_study = H.groupby("StudyInstanceUID").Laterality.agg(lambda s: s.dropna().nunique())
print("studies with conflicting Laterality across series:", int((per_study > 1).sum()))

print("\n=== scanners ===")
print(H.Manufacturer.value_counts(dropna=False).head(8).to_dict())
print("field strength:", H.MagneticFieldStrength.value_counts(dropna=False).head(6).to_dict())
print("transfer syntax:", H.TransferSyntaxUID.value_counts(dropna=False).to_dict())
print("3D acquisitions:", pct(H.MRAcquisitionType == "3D"))

print("\n=== slices per series ===")
print(H.n_slices.describe(percentiles=[.05,.25,.5,.75,.95,.99]).round(1).to_dict())

print("\n=== is the organisers' Fluid_Sensitive flag misfiling T2 without fat-sat? ===")
te = pd.to_numeric(H.EchoTime, errors="coerce"); tr = pd.to_numeric(H.RepetitionTime, errors="coerce")
desc = H.SeriesDescription.fillna("").str.lower(); so = H.ScanOptions.fillna("")
fs_hdr = so.str.contains("FS") | desc.str.contains(r"fs|fat|spair|spir|stir|\bwe\b|_we_|dixon|water")
t2_nofs = (te >= 60) & (tr >= 1500) & ~fs_hdr
sub = H[t2_nofs]
print(f"series that are T2 (TE>=60, TR>=1500) with no fat-sat in the header: {len(sub)} ({pct(t2_nofs)})")
print("  of those, the CSV calls fluid_sensitive=0:", pct(sub.csv_fluid == 0))
print("  studies containing at least one:", sub.StudyInstanceUID.nunique())
print("  by plane:", sub.plane.value_counts().to_dict())

print("\n=== PatientID - do patients repeat across studies? ===")
pid = H.drop_duplicates("StudyInstanceUID").PatientID
print("studies with a PatientID:", pct(pid.notna()), "| distinct:", pid.nunique(), "of", len(pid))
rep = pid.value_counts(); print("PatientIDs appearing in >1 study:", int((rep > 1).sum()))


## Send back

The printed output of the last cell, and download `header_census.csv` into the
local `data/` folder (gitignored). The misfiling count decides whether recovering
the fluid axis from `EchoTime` is worth building into the slot selection.
